# About this notebook

The pipeline groups CEMS units and EIA generators into subplants, using the EPA's power sector data crosswalk (through PUDL) and the manual additions in `reference_tables/epa_eia_crosswalk_manual.csv`. The EPA crosswalk is incomplete and in some cases has incorrect crosswalks. A CEMS unit that is not mapped to any EIA generator becomes a subplant of its own, so its data cannot be combined with the EIA data of the generators it belongs to. Ideally, we would contribute fixes directly to the crosswalk's github repository, but the code is written in R and not python. As a temporary measure, this notebook helps find these gaps and fix them in two reference tables:
- `reference_tables/epa_eia_crosswalk_manual.csv`: new rows mapping CEMS units to EIA generators
- `reference_tables/plants_not_connected_to_grid.csv`: plants whose CEMS units are not connected to the grid

How to use this notebook:
1. Run the pipeline for `year`, so that `outputs/{year}/subplant_crosswalk_{year}.csv.zip` exists (it is written when subplants are identified, in step 2 of the pipeline).
2. **Load the data** (section 1). This also defines `investigate_plant()` and `find_plants()`, which show the data you need to match a plant's CEMS units to its EIA generators. Each table of flagged units below is followed by a cell that calls them, so you can investigate a unit right where it is flagged.
3. **Map the unmapped CEMS units of `year`** (section 2), using matches suggested by boiler ID, or by investigating the plant.
4. **Review the matches suggested by CAMPD** (section 3) for the unmapped units of every year, and investigate the units that CAMPD cannot match.
5. **Review units from non-utility source categories** (section 4), to check whether any CEMS units are not connected to the grid.
6. **Check the updates** (section 5) by rerunning the pipeline and this notebook.

Set `year` to the year whose CEMS data and subplant crosswalk you want to check.

In [ ]:
# import packages
import pandas as pd

%reload_ext autoreload
%autoreload 2

import oge.load_data as load_data
from oge.filepaths import *
import oge.data_cleaning as data_cleaning
import oge.download_data as download_data
from oge.constants import earliest_data_year, current_early_release_year

year = 2025

## 1. Load the data
Run all of the cells in this section before the rest of the notebook.

The CEMS data of `year`, without plants that are not connected to the grid, summed for each unit, with the generators that each unit is mapped to in the subplant crosswalk. A unit with a missing `generator_id` is not mapped to any generator. The subplant crosswalk covers the CEMS units of every year, not only `year`.

In [ ]:
unit_keys = ["plant_id_eia", "emissions_unit_id_epa"]

# load cems data
cems = load_data.load_cems_data(year)
# remove non-grid connected plants
cems = data_cleaning.remove_plants(
    cems,
    year,
    non_grid_connected=True,
    remove_states=["PR"],
    remove_steam_only=False,
    distribution_connected_plants=False,
)

# load the subplant crosswalk, which maps the CEMS units of every year to generators
# NOTE: you will need to run this in the pipeline first
subplant_crosswalk = pd.read_csv(
    outputs_folder(f"{year}/subplant_crosswalk_{year}.csv.zip"),
    dtype={"emissions_unit_id_epa": "str", "generator_id": "str"},
)
mapped_units = (
    subplant_crosswalk.dropna(subset="generator_id").set_index(unit_keys).index
)

# sum operational data by unit and merge the generators of each unit
cems_units = (
    cems.groupby(["plant_id_epa", "plant_id_eia", "emissions_unit_id_epa"])[
        [
            "gross_generation_mwh",
            "fuel_consumed_mmbtu",
            "steam_load_lbs",
            "co2_mass_lb",
        ]
    ]
    .sum()
    .reset_index()
)
cems_units = cems_units.merge(
    subplant_crosswalk.dropna(subset="emissions_unit_id_epa")[
        unit_keys + ["generator_id"]
    ].drop_duplicates(),
    how="left",
    on=unit_keys,
)

The EIA-860 generators of every year, with the last year each generator was reported (`report_date`), the location of its plant, and its boilers. Also the EIA-923 generation and fuel allocated to each generator in `year`.

In [ ]:
# EIA-860 generators of every year, keeping the last year each generator was reported
gens860 = (
    load_data.load_pudl_table(
        "core_eia860__scd_generators",
        columns=[
            "plant_id_eia",
            "generator_id",
            "report_date",
            "operational_status",
            "current_planned_generator_operating_date",
            "prime_mover_code",
            "capacity_mw",
            "energy_source_code_1",
            "energy_source_code_2",
        ],
    )
    .sort_values("report_date")
    .drop_duplicates(subset=["plant_id_eia", "generator_id"], keep="last")
)
plants_entity_eia = load_data.load_pudl_table(
    "core_eia__entity_plants",
    columns=[
        "plant_id_eia",
        "plant_name_eia",
        "street_address",
        "city",
        "county",
        "state",
    ],
)
gens_to_boilers = load_data.load_pudl_table(
    "core_eia860__assn_boiler_generator",
    columns=["plant_id_eia", "generator_id", "boiler_id"],
).drop_duplicates()

gens860 = gens860.merge(
    plants_entity_eia, how="left", on="plant_id_eia", validate="m:1"
).merge(
    gens_to_boilers, how="left", on=["plant_id_eia", "generator_id"], validate="1:m"
)

gen_fuel_allocated = (
    load_data.load_pudl_table(
        "out_eia923__monthly_generation_fuel_by_generator_energy_source", year
    )
    .groupby(
        ["plant_id_eia", "generator_id", "prime_mover_code", "energy_source_code"]
    )[["net_generation_mwh", "fuel_consumed_mmbtu"]]
    .sum()
    .reset_index()
)

The CAMPD units of every year (downloading any facility files that are missing), with one row per unit and associated generator. `first_year` and `last_year` give the years in which CAMPD associates the unit with the generator.

In [ ]:
# load the CAMPD units of every year
campd_units = []
for campd_year in range(earliest_data_year, current_early_release_year + 1):
    download_data.download_campd_facility(campd_year)
    campd_units.append(
        load_data.load_campd_facility(campd_year).assign(year=campd_year)
    )
campd_units = pd.concat(campd_units, ignore_index=True)

# one row per unit and associated generator, with the years they were associated
campd_generators = (
    campd_units.set_index(
        ["plant_id_epa", "plant_id_eia", "emissions_unit_id_epa", "year"]
    )["associated_generators_and_capacity_epa"]
    .str.extractall(
        r"\s*(?P<generator_id_epa>.+?)\s*\((?P<capacity_mw_epa>[\d.]+)\)\s*(?:,|$)"
    )
    .reset_index()
    .astype({"capacity_mw_epa": float})
    .groupby(
        ["plant_id_epa", "plant_id_eia", "emissions_unit_id_epa", "generator_id_epa"],
        as_index=False,
    )
    .agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        capacity_mw_epa=("capacity_mw_epa", "last"),
    )
)

The units that the pipeline removes from CEMS, because they are not connected to the grid or only produce steam. Sections 3 and 4 skip these units.

In [ ]:
ngc_plants = pd.read_csv(reference_table_folder("plants_not_connected_to_grid.csv"))[
    "Plant ID"
]
camd_excluded = load_data.load_epa_eia_crosswalk_from_raw()
camd_excluded = (
    camd_excluded.loc[
        camd_excluded["epa_match_type"] == "Manual CAMD Excluded", unit_keys
    ]
    .set_index(unit_keys)
    .index
)

steam_units = (
    pd.read_csv(
        reference_table_folder("steam_units_to_remove.csv"),
        dtype={"emissions_unit_id_epa": "str"},
    )
    .set_index(unit_keys)
    .index
)

**Investigation helpers.** `investigate_plant(plant_id)` shows the data you need to match the CEMS units of a plant to its EIA generators: the plant's CEMS units with data in `year`, its units and generators in the subplant crosswalk, its CAMPD units and their associated generators, its EIA-860 generators and boilers, and the EIA-923 generation and fuel allocated to its generators in `year`. Look for generator or boiler IDs that resemble a unit's ID, CAMPD capacities that match EIA-860 capacities, and generators whose prime mover, fuel and generation are consistent with the unit's CEMS data.

`find_plants(name, county, city)` finds EIA-860 plants by part of their name, and by county or city. Use it when a plant has no suitable generators, since the unit may report to EIA under another plant ID, for example when a plant was split.

When you find a match, add a row for each generator to `epa_eia_crosswalk_manual.csv`, with a `notes` value describing how you matched it (for example, "Matched based on generator ID"). Fill `start_year` and `end_year` if the mapping only applies to some years, for example when a plant changes its plant ID.

In [ ]:
def investigate_plant(plant_id: int):
    """Shows the data that helps to match the CEMS units of a plant to its EIA
    generators."""
    print(f"CEMS units with data in {year}, and their generators:")
    display(cems_units[cems_units["plant_id_eia"] == plant_id])
    print("Units and generators in the subplant crosswalk:")
    display(
        subplant_crosswalk.loc[
            subplant_crosswalk["plant_id_eia"] == plant_id,
            [
                "emissions_unit_id_epa",
                "generator_id",
                "subplant_id",
                "prime_mover_code",
            ],
        ]
    )
    print("CAMPD units and their associated generators:")
    display(campd_generators[campd_generators["plant_id_eia"] == plant_id])
    print("EIA-860 generators and their boilers:")
    display(gens860[gens860["plant_id_eia"] == plant_id])
    print(f"EIA-923 generation and fuel allocated to each generator in {year}:")
    display(gen_fuel_allocated[gen_fuel_allocated["plant_id_eia"] == plant_id])


def find_plants(name: str = None, county: str = None, city: str = None):
    """Finds EIA-860 plants by part of their name, and by county or city."""
    plants = gens860[
        ["plant_id_eia", "plant_name_eia", "street_address", "city", "county", "state"]
    ].drop_duplicates()
    if name is not None:
        plants = plants[
            plants["plant_name_eia"].str.contains(name, case=False, na=False)
        ]
    if county is not None:
        plants = plants[plants["county"] == county]
    if city is not None:
        plants = plants[plants["city"] == city]
    return plants

## 2. Map the unmapped CEMS units of `year`
**Unmapped units with generation.** These matter most, since their generation and emissions are not combined with the EIA data of their plant.

In [ ]:
# identify units that are not mapped to a generator and have non-zero generation
cems_units[cems_units["generator_id"].isna() & (cems_units["gross_generation_mwh"] > 0)]

**Unmapped units without generation.** These are often units that only produce steam, or that did not operate. They matter less, but units that report fuel or emissions should still be mapped. If a unit only produces steam and does not exist in EIA, add it to `steam_units_to_remove.csv` instead (see `update_steam_units_to_remove.ipynb`).

In [ ]:
# identify units that are not mapped to a generator and have zero generation
cems_units[
    cems_units["generator_id"].isna() & (cems_units["gross_generation_mwh"] == 0)
]

**Matches suggested by boiler ID.** Some CEMS unit IDs are EIA boiler IDs. For each unmapped unit above, this lists the EIA-860 generators associated with a boiler that has the same ID as the unit. If a match looks right, add a row for each generator to `epa_eia_crosswalk_manual.csv`, with the note "Matched based on boiler_id and EIA-860 BGA table".

In [ ]:
match = {}
for i in cems_units[cems_units["generator_id"].isna()].index:
    plant_id = cems_units.loc[i, "plant_id_eia"]
    emission_unit_id = cems_units.loc[i, "emissions_unit_id_epa"]
    df = gens860.query(
        "plant_id_eia == @plant_id and boiler_id == @emission_unit_id"
    ).drop_duplicates(keep="first")
    if not df.empty:
        match[(plant_id, emission_unit_id)] = df

match

**Investigate a unit.** For the units above that are not matched by boiler ID, and not matched by CAMPD in section 3, set the plant ID to the unit's plant (the example is Eastman Chemical Company, 50481).

In [ ]:
campd_units[campd_units["plant_id_epa"] == 50481]

In [ ]:
investigate_plant(50481)

If the plant has no suitable generators, look for plants with a similar name in the same county or city, updating the arguments for your plant.

In [ ]:
display(find_plants(name="Eastman", county="Sullivan"))
display(find_plants(city="Kingsport"))

## 3. Review the matches suggested by CAMPD
The CAMPD facility file of each year lists the generators associated with each unit. These generator IDs are CAMD's labels, and do not always match the EIA generator IDs. So we only suggest a match when every generator associated with a unit exists in EIA-860 at the plant, after ignoring spaces, dashes, underscores, letter case, and leading zeros (skipping EIA-860 generator IDs that are no longer unique at the plant once these are ignored).

Unlike section 2, this covers the unmapped units of every year in the subplant crosswalk, not only the units with data in `year`. `generator_id` is the matched EIA-860 generator, and is missing when there is no match. `all_generators_in_eia860` is `True` when every CAMPD generator of the unit is matched.

In [ ]:
# CEMS units of every year that are not mapped to any generator in the subplant crosswalk
unmapped_units = subplant_crosswalk.dropna(subset="emissions_unit_id_epa")[
    unit_keys
].drop_duplicates()
unmapped_units = unmapped_units[
    ~unmapped_units.set_index(unit_keys).index.isin(mapped_units)
]


# match the CAMPD generators to the EIA-860 generators of any year
def normalize_generator_id(generator_id):
    return (
        generator_id.str.replace(r"[\s\-_]", "", regex=True).str.upper().str.lstrip("0")
    )


eia860_generators = gens860[
    [
        "plant_id_eia",
        "generator_id",
        "prime_mover_code",
        "capacity_mw",
        "operational_status",
    ]
].drop_duplicates()
eia860_generators["generator_id_normalized"] = normalize_generator_id(
    eia860_generators["generator_id"]
)
# skip generator IDs that are no longer unique at the plant once normalized
eia860_generators = eia860_generators.drop_duplicates(
    subset=["plant_id_eia", "generator_id_normalized"], keep=False
)

crosswalk_candidates = unmapped_units.merge(campd_generators, how="inner", on=unit_keys)
crosswalk_candidates["generator_id_normalized"] = normalize_generator_id(
    crosswalk_candidates["generator_id_epa"]
)
crosswalk_candidates = crosswalk_candidates.merge(
    eia860_generators,
    how="left",
    on=["plant_id_eia", "generator_id_normalized"],
    validate="m:1",
)
crosswalk_candidates["all_generators_in_eia860"] = crosswalk_candidates.groupby(
    unit_keys
)["generator_id"].transform(lambda generator_id: generator_id.notna().all())

**Units that CAMPD fully matches.** Check that each match makes sense before adding it:
- `capacity_mw_epa` should be close to the EIA-860 `capacity_mw`. A large difference can mean the wrong generator, although it can also come from several boilers sharing one turbine, or from different capacity definitions.
- `prime_mover_code` should fit the unit, and a unit with several generators should usually be a combined cycle (`CT` and `CA`) or several boilers feeding one turbine.

To check a match more closely, use `investigate_plant()` in the next cell. Note the units whose match you do not accept.

In [ ]:
crosswalk_candidates[crosswalk_candidates["all_generators_in_eia860"]]

In [ ]:
investigate_plant(55098)

**Rows to add to `epa_eia_crosswalk_manual.csv`.** This prints the fully matched units as CSV rows, skipping units that the pipeline removes because they are not connected to the grid. Delete the rows of any units you did not accept, and paste the rest at the end of `reference_tables/epa_eia_crosswalk_manual.csv`.

In [ ]:
new_manual_rows = (
    crosswalk_candidates[
        crosswalk_candidates["all_generators_in_eia860"]
        & ~crosswalk_candidates["plant_id_eia"].isin(ngc_plants)
        & ~crosswalk_candidates.set_index(unit_keys).index.isin(camd_excluded)
    ]
    .assign(start_year=pd.NA, end_year=pd.NA, notes="CAMPD Facility database mapping")[
        [
            "plant_id_epa",
            "emissions_unit_id_epa",
            "plant_id_eia",
            "generator_id",
            "start_year",
            "end_year",
            "notes",
        ]
    ]
    .sort_values(["plant_id_epa", "emissions_unit_id_epa", "generator_id"])
)

print(new_manual_rows.to_csv(index=False, header=False))

**Units that CAMPD cannot fully match.** Some or all of the CAMPD generator IDs of these units do not exist in EIA-860 (for example, CAMPD lists `GEPS-1` where EIA uses `1`), so they need to be matched manually. Many of these units retired long ago, so check `last_year`, and use `generator_id_epa` and `capacity_mw_epa` as hints when investigating them in the next cell.

In [ ]:
crosswalk_candidates[~crosswalk_candidates["all_generators_in_eia860"]]

In [ ]:
investigate_plant(2504)

## 4. Review units from non-utility source categories
CAMPD assigns each unit a source category. Units in non-utility categories (such as industrial boilers, refineries, and pulp and paper mills) may not be connected to the grid. This lists the units in `year` with a non-utility source category, skipping units that the pipeline already removes, with their CEMS output, whether they are mapped to a generator, and the EIA data of their plant.

What to look for:
- Plants with EIA-923 net generation (`net_generation_mwh`) and a balancing authority are connected to the grid. Most industrial cogeneration plants are like this, and should be kept.
- Units with steam output but no generation (`steam_load_lbs` above zero and `gross_generation_mwh` of zero) are handled by the pipeline's checks for steam-only units, and do not need to be added here.
- Units with CEMS generation at a plant without EIA-923 data or a balancing authority may not be connected to the grid. Investigate them in the next cell, and confirm that they are not connected to the grid before adding the plant to `plants_not_connected_to_grid.csv` (with its `Plant ID`, `State` and `Plant Name`), since this removes all of the plant's data from OGE.

In [ ]:
campd_units_year = load_data.load_campd_facility(year)
nonutility_units = campd_units_year[
    ~campd_units_year["source_category_epa"].isin(
        ["Electric Utility", "Cogeneration", "Small Power Producer"]
    )
]

# skip units that are already removed
nonutility_units = nonutility_units[
    ~nonutility_units["plant_id_eia"].isin(ngc_plants)
    & ~nonutility_units["plant_id_eia"].between(880000, 889999)
    & ~nonutility_units.set_index(unit_keys).index.isin(camd_excluded)
    & ~nonutility_units.set_index(unit_keys).index.isin(steam_units)
]

# add the CEMS output of each unit, and the EIA data of each plant
eia_plants = load_data.load_pudl_table(
    "out_eia__yearly_plants",
    year,
    columns=["plant_id_eia", "sector_name_eia", "balancing_authority_code_eia"],
)
eia923_generation = (
    load_data.load_pudl_table(
        "out_eia923__generation_fuel_combined",
        year,
        columns=["plant_id_eia", "net_generation_mwh"],
    )
    .groupby("plant_id_eia", as_index=False)
    .sum()
)
nonutility_units = (
    nonutility_units[
        [
            "plant_id_eia",
            "plant_name_epa",
            "state",
            "source_category_epa",
            "emissions_unit_id_epa",
            "unit_type_epa",
        ]
    ]
    .merge(
        cems.groupby(unit_keys, as_index=False)[
            ["gross_generation_mwh", "steam_load_lbs", "fuel_consumed_mmbtu"]
        ].sum(),
        how="left",
        on=unit_keys,
        validate="1:1",
    )
    .assign(
        mapped_to_generator=lambda df: df.set_index(unit_keys).index.isin(mapped_units)
    )
    .merge(eia_plants, how="left", on="plant_id_eia", validate="m:1")
    .merge(eia923_generation, how="left", on="plant_id_eia", validate="m:1")
)
nonutility_units

In [ ]:
investigate_plant(10698)

## 5. Check the updates
After updating the reference tables, rerun the pipeline for `year`, so that subplants are identified again with the new mappings, and rerun this notebook. The units you mapped should no longer appear in sections 2 and 3, and the plants you added to `plants_not_connected_to_grid.csv` should no longer appear in sections 2 and 4.